In [38]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect(
    "memory.db", 
    check_same_thread=False,
)

config = {
    "configurable": {
        "thread_id": "1"
    }
}

In [39]:
class State(MessagesState):
    pass

graph_builder = StateGraph(State)

In [40]:
@tool
def get_human_feedback(poem: str):
    """
    Asks the user for feedback on the poem.
    Use this before returning the final response.
    """
    feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")
    return feedback


llm_with_tools = llm.bind_tools([get_human_feedback])

def chatbot(state: State):
    response = llm_with_tools.invoke(
        f"""
        You are an expert in making poems.

        Use the `get_human_feedback` tool to get feedback on your poem.

        Only after you receive positive feedback you can return the final poem.

        ALWAYS ASK FOR FEEDBACK FIRST.

        Here is the conversation history:

        {state["messages"]}
    """
    )
    return {
        "messages": [response],
    }

In [41]:
tool_node = ToolNode(
    tools=[
        get_human_feedback,
    ],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [42]:
result = graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "Please make a poem about Python code."},
        ]
    },
    config=config,
)

In [43]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_DkVz6KYigZbmmZO1ktpdypcA)
 Call ID: call_DkVz6KYigZbmmZO1ktpdypcA
  Args:
    poem: In the realm of code where logic flows,
Python whispers secrets only it knows.
Indent with care, let the functions sing,
A symphony of syntax, a digital spring.

Variables dance, in memory they play,
Looping through data, both night and day.
From lists to dictionaries, endless arrays,
Crafting solutions in myriad ways.

With libraries strong, like NumPy and more,
We explore the vastness, we open the door.
Data unfolds like a beautiful tale,
In Python's embrace, we’re destined to sail.

So here’s to the coders, with coffee in hand,
Building the future, both simple and grand.
In the heart of Python, creativity glows,
A canvas of code, where imagination flows.


In [44]:
snapshot = graph.get_state(config)

snapshot.next

('tools',)

In [45]:
response = Command(resume="It looks great!")

result = graph.invoke(
    response,
    config=config,
)

In [46]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_DkVz6KYigZbmmZO1ktpdypcA)
 Call ID: call_DkVz6KYigZbmmZO1ktpdypcA
  Args:
    poem: In the realm of code where logic flows,
Python whispers secrets only it knows.
Indent with care, let the functions sing,
A symphony of syntax, a digital spring.

Variables dance, in memory they play,
Looping through data, both night and day.
From lists to dictionaries, endless arrays,
Crafting solutions in myriad ways.

With libraries strong, like NumPy and more,
We explore the vastness, we open the door.
Data unfolds like a beautiful tale,
In Python's embrace, we’re destined to sail.

So here’s to the coders, with coffee in hand,
Building the future, both simple and grand.
In the heart of Python, creativity glows,
A canvas of code, where imagination flows.
==========